# 因子1: 盘口压力不对称因子 (Order Book Pressure Asymmetry)

- **数据源**: `bigalpha_2026_stock_bar1m` (从 `datasources['bar1m']` 动态获取)
- **经济逻辑**: 盘口买卖挂单量的不对称反映市场参与者真实意图。买方挂单远大于卖方时，表明市场有强烈需求支撑，未来收益更可能为正。通过5档加权(近档权重高)计算可捕捉更深层次市场压力。
- **正交性**: factorlib仅有日频主力资金流(netflow_amount_main)，本因子直接从微观盘口结构提取信号，信息维度完全不同。
- **传统赛道标注**: 核心统计方法为加权盘口压力比 + 5日时序平滑；经济逻辑实现位置在 main 函数 SQL 内。

In [6]:
def main(datasources, start_date, end_date):
    import pandas as pd
    import dai
    
    # ============================================================
    # 盘口压力不对称因子
    # 经济逻辑: 加权买方挂单 - 加权卖方挂单, 标准化后5日平滑
    # 统计方法: 5档线性加权(5,4,3,2,1) + 日内均值 + 5日移动平均
    # ============================================================
    
    # 从 datasources 字典获取表名 (不能硬编码)
    bar1m = datasources["bar1m"]
    
    # 时序算子需要历史数据, 往前多取缓冲天数
    BUFFER_DAYS = 20
    query_start = (pd.to_datetime(start_date) - pd.Timedelta(days=BUFFER_DAYS)).strftime('%Y-%m-%d %H:%M:%S')
    
    sql = f"""
    WITH cte_daily AS (
        SELECT
            instrument,
            strftime(date, '%Y-%m-%d') AS trading_day,
            AVG(
                (
                    bid_volume1 * 5.0 + bid_volume2 * 4.0 + bid_volume3 * 3.0 +
                    bid_volume4 * 2.0 + bid_volume5 * 1.0
                    -
                    ask_volume1 * 5.0 - ask_volume2 * 4.0 - ask_volume3 * 3.0 -
                    ask_volume4 * 2.0 - ask_volume5 * 1.0
                ) /
                NULLIF(
                    bid_volume1 + bid_volume2 + bid_volume3 + bid_volume4 + bid_volume5 +
                    ask_volume1 + ask_volume2 + ask_volume3 + ask_volume4 + ask_volume5,
                    0
                )
            ) AS daily_pressure
        FROM {bar1m}
        WHERE close > 0
        GROUP BY instrument, strftime(date, '%Y-%m-%d')
    ),
    cte_smoothed AS (
        SELECT
            *,
            AVG(daily_pressure) OVER (
                PARTITION BY instrument ORDER BY trading_day ROWS 4 PRECEDING
            ) AS smoothed_pressure
        FROM cte_daily
    )
    SELECT
        CAST(trading_day AS DATETIME) AS date,
        instrument,
        smoothed_pressure AS factor
    FROM cte_smoothed
    """
    
    # 用扩过的下界查询
    df = dai.query(
        sql,
        filters={'date': [query_start, end_date]},
        compression=True,
    ).df()
    
    # 裁回真正的评估区间
    df = df[(df['date'] >= pd.to_datetime(start_date)) & (df['date'] <= pd.to_datetime(end_date))]
    
    # 用股票池过滤 (中证1000成分股, 此表不替换, 直接写表名)
    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={'date': [start_date, end_date]},
    ).df()
    df = pd.merge(df, stk_pool, how='inner', on=['date', 'instrument'])
    
    return df

In [5]:
# ============================================================
# 本地测试: 调用 main() 生成因子数据并检查
# ============================================================

# --- 方式1: 训练集测试 ---
datasources = {
    'bar1m': 'bigalpha_2026_stock_bar1m',
    'financial': 'bigalpha_2026_financial'
}
start_date = '2019-01-01 00:00:00'
end_date = '2023-12-31 23:59:59'

# --- 方式2: 自检表模拟验证集 (取消注释即可) ---
# datasources = {'bar1m': 'bigalpha_2026_stock_bar1m_selftest', 'financial': 'bigalpha_2026_financial'}
# start_date = '2024-01-01 00:00:00'
# end_date = '2024-10-31 23:59:59'

factor_data = main(datasources, start_date, end_date)

print("=== 因子数据检查 ===")
print(f"列名: {list(factor_data.columns)}")
print(f"行数: {len(factor_data)}")
print(f"\n前5行:")
print(factor_data.head())
print(f"\n因子统计:")
print(factor_data['factor'].describe())
print(f"\n缺失值: {factor_data['factor'].isna().sum()}")
print(f"inf值: {factor_data['factor'].isin([float('inf'), float('-inf')]).sum()}")

# 覆盖度检查: 每日缺失率
print("\n=== 覆盖度检查 ===")
daily_coverage = factor_data.groupby('date')['factor'].agg(['count', 'size'])
daily_coverage['missing_rate'] = 1 - daily_coverage['count'] / daily_coverage['size']
high_missing = daily_coverage[daily_coverage['missing_rate'] > 0.4]
if len(high_missing) > 0:
    print(f"⚠️ 有 {len(high_missing)} 个交易日缺失率 > 40%:")
    print(high_missing.head())
else:
    print("✅ 所有交易日缺失率 <= 40%")

=== 因子数据检查 ===
列名: ['date', 'instrument', 'factor']
行数: 1209218

前5行:
        date instrument    factor
0 2019-01-02  000034.SZ       NaN
1 2019-01-03  000034.SZ       NaN
2 2019-01-04  000034.SZ       NaN
3 2019-01-07  000034.SZ       NaN
4 2019-01-08  000034.SZ  1.085616

因子统计:
count    1.205222e+06
mean     2.453772e-01
std      3.912838e-01
min     -4.991324e+00
25%      2.357304e-02
50%      2.284482e-01
75%      4.504299e-01
max      4.990681e+00
Name: factor, dtype: float64

缺失值: 3996
inf值: 0

=== 覆盖度检查 ===
⚠️ 有 4 个交易日缺失率 > 40%:
            count  size  missing_rate
date                                 
2019-01-02      0   998           1.0
2019-01-03      0   998           1.0
2019-01-04      0   998           1.0
2019-01-07      0   998           1.0
